# 01 · Exploring audio features

A playground for Phase 2. Run each cell (Shift+Enter), change numbers, break things.
Nothing here affects the app; the real code lives in `src/musiccompare/features/`.

Start it with: `uv run jupyter lab` and open this file.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt  # fine in notebooks; the app itself avoids pyplot
import numpy as np
from IPython.display import Audio

from musiccompare import config
from musiccompare.features.extract import extract_features
from musiccompare.models import AudioClip

SR = config.SAMPLE_RATE

## 1. Make a test melody

A short 'viola-like' phrase: each note is a fundamental plus a few overtones,
with a gentle swell in loudness. Try changing the notes or adding `vibrato_cents`.

In [ ]:
def note(midi, dur=0.5, vibrato_cents=0.0, overtones=(1.0, 0.5, 0.3, 0.2)):
    t = np.arange(int(dur * SR)) / SR
    f0 = librosa.midi_to_hz(midi) * 2 ** (vibrato_cents * np.sin(2 * np.pi * 5.5 * t) / 1200)
    phase = 2 * np.pi * np.cumsum(f0) / SR
    y = sum(a * np.sin((h + 1) * phase) for h, a in enumerate(overtones))
    envelope = np.minimum(1, t / 0.03) * np.minimum(1, (dur - t) / 0.05)  # soft attack/release
    return y * envelope


melody_midi = [60, 62, 64, 65, 67, 65, 64, 62, 60]  # C D E F G F E D C
y = np.concatenate([note(m) for m in melody_midi])
y *= np.linspace(0.3, 1.0, len(y))  # crescendo
y = (0.8 * y / np.abs(y).max()).astype(np.float32)
clip = AudioClip(y=y, sr=SR, name="melody")
Audio(y, rate=SR)

## 2. The spectrogram (STFT)

The STFT slices audio into overlapping frames and measures how much energy each
frequency has in each frame. Horizontal stripes = a note's fundamental and overtones.

In [ ]:
S_db = librosa.amplitude_to_db(
    np.abs(librosa.stft(y, n_fft=config.FRAME_LENGTH, hop_length=config.HOP_LENGTH)), ref=np.max
)
fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(
    S_db, sr=SR, hop_length=config.HOP_LENGTH, x_axis="time", y_axis="log", ax=ax
)
fig.colorbar(img, ax=ax, format="%+0.0f dB")
ax.set_title("Log-frequency spectrogram");

## 3. All features at once

`extract_features` returns a `FeatureSet`: one value per frame for each feature.

In [ ]:
fs = extract_features(clip)
print(f"{fs.n_frames} frames, one every {config.HOP_LENGTH / SR * 1000:.1f} ms")

panels = [
    ("MIDI note", librosa.hz_to_midi(fs.f0_hz)),
    ("Loudness (dB)", fs.rms_db),
    ("Centroid (Hz)", fs.centroid_hz),
    ("Flatness", fs.flatness),
]
fig, axes = plt.subplots(len(panels), 1, figsize=(10, 9), sharex=True)
for ax, (label, values) in zip(axes, panels, strict=True):
    ax.plot(fs.times, values)
    ax.set_ylabel(label)
axes[-1].set_xlabel("Time (s)");

## 4. Experiments to try

1. Add `vibrato_cents=30` to the notes. What happens to the pitch curve?
2. Set `overtones=(1.0,)` (a pure sine). How does the centroid change? Why?
3. Add noise: `y = y + 0.05 * np.random.default_rng(0).standard_normal(len(y))`. Watch flatness.
4. Load a real recording: `from musiccompare.audio.loader import load_audio; clip = load_audio("path/to/file.wav")`.